### RUN IN CPU ~00:05:00

In [1]:
# CELL 1
import time

start_time = time.time()

In [2]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    rl_root = Path("/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2")
    !pip install -q pyarrow fastparquet
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

In [3]:
# CELL 3: Load Data, Filter & Align Dates
import pandas as pd
import numpy as np
from core.settings import TradingConfig
from core.paths import GLOBAL_PROCESSED_DIR, LOCAL_DATA_DIR
from data_pipeline.loader import load_raw_global_data
from data_pipeline.utils import get_master_trading_calendar
from data_pipeline.builder import generate_features

pd.set_option("future.no_silent_downcasting", True)
config = TradingConfig()

print("Loading raw global data...")
df_ohlcv, df_fed, df_indices = load_raw_global_data()

# ---> FIX: Type assertions for Pylance <---
assert isinstance(df_ohlcv, pd.DataFrame), "df_ohlcv must be a pandas DataFrame"
assert isinstance(df_ohlcv.index, pd.MultiIndex), "df_ohlcv index must be a MultiIndex"

# ---------------------------------------------------------
# OPTIMIZATION 1: Truncate Dates Early
# We train from 1998, but need ~1 year (252 days) for rolling features.
# ---------------------------------------------------------
CUTOFF_DATE = pd.Timestamp("1997-01-01")
df_ohlcv = df_ohlcv[df_ohlcv.index.get_level_values("Date") >= CUTOFF_DATE]

print(f"Aligning to Master Trading Calendar ({config.calendar_ticker})...")
master_dates = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)
master_dates = master_dates[master_dates >= CUTOFF_DATE]

# Filter out dates that are not in the master calendar
df_ohlcv = df_ohlcv[df_ohlcv.index.get_level_values("Date").isin(master_dates)]

# ---------------------------------------------------------
# OPTIMIZATION 2: Fast Pre-Screening (Shift Left)
# Drop stocks that NEVER meet the $1M median dollar volume threshold
# to save RAM during feature generation.
# ---------------------------------------------------------
print("Pre-screening universe to drop perpetually illiquid stocks...")
df_ohlcv["DollarVolume"] = df_ohlcv["Adj Close"] * df_ohlcv["Volume"]

# Calculate rolling 126d median (as per config.quality_min_periods)
# and find the max value it ever reached in its history.
max_med_dv = (
    df_ohlcv.groupby(level="Ticker")["DollarVolume"]
    .rolling(window=config.quality_min_periods, min_periods=30)
    .median()
    .groupby(level=0)  # Use level=0 to avoid the duplicate "Ticker" name error
    .max()
)

# Keep only tickers that breached the $1M threshold at least once
valid_tickers = max_med_dv[
    max_med_dv >= config.thresholds.min_median_dollar_volume
].index

# ---> FIX: PROTECT THE BENCHMARK & CALENDAR TICKERS <---
protected_tickers = pd.Index([config.calendar_ticker, config.benchmark_ticker])
valid_tickers = valid_tickers.union(protected_tickers)

df_ohlcv = df_ohlcv[
    df_ohlcv.index.get_level_values("Ticker").isin(valid_tickers)
].copy()
df_ohlcv.drop(columns=["DollarVolume"], inplace=True)
print(
    f"Universe reduced to {len(valid_tickers)} viable tickers (Benchmarks protected)."
)


# ---------------------------------------------------------
# OPTIMIZATION 3: Smart Forward Fill (No Cartesian Product)
# Reindex each stock ONLY within its own valid date range (IPO to Delist)
# ---------------------------------------------------------
def fill_ticker(df_ticker):
    # Find the min and max date for THIS specific stock
    min_date = df_ticker.index.get_level_values("Date").min()
    max_date = df_ticker.index.get_level_values("Date").max()

    # Create a calendar just for this stock's lifespan
    stock_calendar = master_dates[
        (master_dates >= min_date) & (master_dates <= max_date)
    ]

    # Reindex and forward fill gaps (up to max_data_gap_ffill)
    df_ticker = df_ticker.droplevel("Ticker").reindex(stock_calendar)

    price_cols = ["Adj Open", "Adj High", "Adj Low", "Adj Close"]
    if config.handle_zeros_as_nan:
        df_ticker[price_cols] = df_ticker[price_cols].replace(0, np.nan)

    df_ticker[price_cols] = df_ticker[price_cols].ffill(limit=config.max_data_gap_ffill)
    df_ticker["Volume"] = df_ticker["Volume"].fillna(0)

    return df_ticker


print("Forward-filling intra-lifespan data gaps...")
# Let Pandas naturally reconstruct the MultiIndex
df_ohlcv = df_ohlcv.groupby(level="Ticker").apply(fill_ticker)


# ==============================================================================
# INJECT SYNTHETIC "CASH" ASSET
# ==============================================================================
print("Injecting synthetic CASH asset for RL environment...")

# 1. Use the exact master calendar so CASH never has missing days
cash_dates = master_dates
np.random.seed(42)  # Keep the noise deterministic across pipeline runs

# 2. Generate mean-reverting micro-noise (6th decimal place)
# This perfectly centers RSI at 50 and creates a tiny, safe variance
price_noise = np.random.normal(0, 1e-6, len(cash_dates))
cash_close = 100.0 + price_noise

# 3. Generate Volume noise so RollingSameVolCount == 0
vol_noise = np.random.randint(-5000, 5000, len(cash_dates))
cash_volume = 100_000_000.0 + vol_noise

# 4. Construct the dataframe
cash_df = pd.DataFrame(
    {
        "Ticker": "CASH",
        "Date": cash_dates,
        "Adj Open": cash_close,
        "Adj High": cash_close
        + 1e-4,  # Strictly > Close to bypass 'adj_high == adj_low' stale trap
        "Adj Low": cash_close - 1e-4,  # Strictly < Close
        "Adj Close": cash_close,
        "Volume": cash_volume,
    }
)

# 5. Format and append
cash_df = cash_df.set_index(["Ticker", "Date"])
df_ohlcv = pd.concat([df_ohlcv, cash_df]).sort_index()

print(f"Successfully injected {len(cash_dates)} rows of synthetic CASH.")


# Ensure index names are correct after apply
if isinstance(df_ohlcv.index, pd.MultiIndex):
    df_ohlcv.index.names = ["Ticker", "Date"]

# Align df_fed & df_indices
if df_fed is not None:
    df_fed["Date"] = pd.to_datetime(df_fed["Date"])
    df_fed = df_fed.set_index("Date").reindex(master_dates).ffill()

if df_indices is not None:
    assert isinstance(df_indices.index, pd.MultiIndex)
    df_indices = df_indices[
        df_indices.index.get_level_values("Date").isin(master_dates)
    ].sort_index()  # Sorts by all levels naturally

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

Loading raw global data...
Aligning to Master Trading Calendar (SPY)...
Pre-screening universe to drop perpetually illiquid stocks...
Universe reduced to 1717 viable tickers (Benchmarks protected).
Forward-filling intra-lifespan data gaps...
Injecting synthetic CASH asset for RL environment...
Successfully injected 7451 rows of synthetic CASH.


In [4]:
print(f"df_ohlcv.info():\n{df_ohlcv.info()}\n")
print(f"df_fed.info():\n{df_fed.info()}\n")
print(f"df_indices.info():\n{df_indices.info()}\n")

print(f"df_ohlcv.describe():\n{df_ohlcv.describe()}\n")
print(f"df_fed.describe():\n{df_fed.describe()}\n")
print(f"df_indices.describe():\n{df_indices.describe()}\n")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 8572304 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-08-14 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     float64
dtypes: float64(5)
memory usage: 360.1+ MB
df_ohlcv.info():
None

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7451 entries, 1997-01-02 to 2026-08-14
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   High_Yield_Spread  753 non-null    float32
 1   Yield_Curve_10Y2Y  7451 non-null   float32
dtypes: float32(2)
memory usage: 116.4 KB
df_fed.info():
None

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 12152 entries, ('^VIX', Timestamp('1997-01-02 00:00:00')) to ('^VIX3M', Timestamp('2026-08-14 00:00:00'))
Data columns (total 6 columns):
 #   Column    

In [5]:
df_ohlcv.loc["CASH"]

,Adj Open,Adj High,Adj Low,Adj Close,Volume
Date,,,,,
1997-01-02,100.000000,100.000100,99.999900,100.000000,100001840.0
1997-01-03,100.000000,100.000100,99.999900,100.000000,100000066.0
1997-01-06,100.000001,100.000101,99.999901,100.000001,99997415.0
1997-01-07,100.000002,100.000102,99.999902,100.000002,99997389.0
1997-01-08,100.000000,100.000100,99.999900,100.000000,100000924.0
...,...,...,...,...,...
2026-08-10,100.000001,100.000101,99.999901,100.000001,100000758.0
2026-08-11,100.000000,100.000100,99.999900,100.000000,99996704.0
2026-08-12,100.000001,100.000101,99.999901,100.000001,100001074.0


In [6]:
import pandas as pd

print("=== Analyzing Ticker Lifespans ===")

# 1. Get the global max and min dates of the whole dataset
global_min_date = df_ohlcv.index.get_level_values("Date").min()
global_max_date = df_ohlcv.index.get_level_values("Date").max()
print(f"Global Dataset Range: {global_min_date.date()} to {global_max_date.date()}")

# 2. Get the specific min and max dates for EACH ticker
ticker_lifespans = (
    df_ohlcv.index.to_frame(index=False).groupby("Ticker")["Date"].agg(["min", "max"])
)
ticker_lifespans.columns = ["First_Trade", "Last_Trade"]

# 3. Identify different categories
# "Delisted" = stopped trading before the global max date
delisted = ticker_lifespans[ticker_lifespans["Last_Trade"] < global_max_date]
# "Late IPO" = started trading after the global min date
late_ipos = ticker_lifespans[ticker_lifespans["First_Trade"] > global_min_date]
# "Came and Went" = Started late AND Delisted early
# ---> FIX: Added .copy() at the end of the slice to prevent SettingWithCopyWarning <---
came_and_went = ticker_lifespans[
    (ticker_lifespans["First_Trade"] > global_min_date)
    & (ticker_lifespans["Last_Trade"] < global_max_date)
].copy()

print(f"\nTotal Tickers: {len(ticker_lifespans)}")
print(f"Tickers that Delisted before end date: {len(delisted)}")
print(f"Tickers that IPO'd after start date: {len(late_ipos)}")
print(f"Tickers that 'Came and Went' (Both): {len(came_and_went)}")

# Display a sample of the ones that "came and went"
if not came_and_went.empty:
    print("\nSample of stocks that 'Came and Went':")
    came_and_went["Lifespan_Days"] = (
        came_and_went["Last_Trade"] - came_and_went["First_Trade"]
    ).dt.days
    print(came_and_went.sort_values("Lifespan_Days").head(10))

=== Analyzing Ticker Lifespans ===
Global Dataset Range: 1997-01-02 to 2026-08-14

Total Tickers: 1717
Tickers that Delisted before end date: 0
Tickers that IPO'd after start date: 1159
Tickers that 'Came and Went' (Both): 0


In [7]:
# CELL 4: Generate Features
print("Generating features_df and macro_df (~6 mins)...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv, df_indices=df_indices, df_fed=df_fed, config=config
)

Generating features_df and macro_df (~6 mins)...
[EXEC] Generating Decoupled Features (Benchmark: SPY)...
[EXEC] Sanitizing features to prevent RL Environment crashes...


In [8]:
print(macro_df.tail(20))

             Mkt_Ret  Mkt_Ret_Z  Macro_Trend  Macro_Trend_Z  \
Date                                                          
2026-07-20 -0.001614  -0.292776     0.069530      -0.419520   
2026-07-21  0.008341   0.903041     0.077788      -0.164344   
2026-07-22 -0.001163  -0.268111     0.075887      -0.222676   
2026-07-23 -0.012349  -1.580407     0.062031      -0.648198   
2026-07-24  0.001016   0.033238     0.062553      -0.628568   
2026-07-27  0.000217  -0.049832     0.062207      -0.635723   
2026-07-28  0.002395   0.215582     0.064191      -0.571014   
2026-07-29 -0.015388  -1.890534     0.047335      -1.082647   
2026-07-30  0.016766   1.856766     0.064176      -0.561912   
2026-07-31  0.007200   0.757339     0.071151      -0.345942   
2026-08-03  0.014243   1.526538     0.085622       0.097819   
2026-08-04  0.018029   1.867937     0.104312       0.666592   
2026-08-05 -0.001997  -0.340849     0.101204       0.567903   
2026-08-06 -0.001598  -0.273412     0.098583       0.48

In [9]:
# CELL 5: Save Datasets
print("Saving Processed Datasets...")

# 1. Save "Production/Daily" copies to the Global Directory for other notebooks
df_ohlcv.to_parquet(GLOBAL_PROCESSED_DIR / "df_ohlcv.parquet")
if df_fed is not None:
    df_fed.to_parquet(GLOBAL_PROCESSED_DIR / "df_fed.parquet")
if df_indices is not None:
    df_indices.to_parquet(GLOBAL_PROCESSED_DIR / "df_indices.parquet")

# 2. Save "Version-Locked Snapshots" to the Local Directory for RLVR Training & Cache
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_ohlcv.to_parquet(LOCAL_DATA_DIR / "df_ohlcv.parquet")
if df_fed is not None:
    df_fed.to_parquet(LOCAL_DATA_DIR / "df_fed.parquet")
if df_indices is not None:
    df_indices.to_parquet(LOCAL_DATA_DIR / "df_indices.parquet")

# 3. Save matching features and macro files to the Local Directory
features_df.to_parquet(LOCAL_DATA_DIR / "features_df.parquet")
macro_df.to_parquet(LOCAL_DATA_DIR / "macro_df.parquet")

print(f"All files exported to LOCAL_DATA_DIR: {LOCAL_DATA_DIR}.")

Saving Processed Datasets...
All files exported to LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data.


In [10]:
# CELL 6: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")

Time: 00:06:51


In [11]:
# CELL 10: Disconnect from TPU
from google.colab import runtime

# This will restart and unassign the current runtime,
# effectively disconnecting from the T4 GPU.
runtime.unassign()
print("Runtime unassigned. This session has been disconnected.")

ModuleNotFoundError: No module named 'google.colab'